## Notebook conventions

**Edit this file in place — don't save a new copy** (no `_V1`/`_V2`/dated/`-Copy`/`DEBUG-` variants). Commit changes via a branch + PR; git history is the version record, not the filename. Full conventions and `nbstripout` setup: see [CONTRIBUTING.md](../../CONTRIBUTING.md) → "Notebook conventions."

# Video Processing Notebook for MegaDetector

This notebook extracts frames from video files and processes them through MegaDetector for wildlife detection.

**System Requirements:**
- Windows OS
- Python 3.9+ (Tarazed runs 3.11)
- MegaDetector installed
- OpenCV

In [ ]:
import os
import json
import shutil
import subprocess
import time
from pathlib import Path
import cv2
import tkinter as tk
from tkinter import filedialog as fd

# MegaDetector imports (from the pip-installed package)
from megadetector.utils import path_utils
from megadetector.detection import video_utils
from megadetector.utils.ct_utils import write_json
from megadetector.visualization import visualization_utils as vis_utils
from megadetector.detection.video_utils import frame_results_to_video_results, FrameToVideoOptions
# Python API for MegaDetector (load the model once, reuse in-process) instead
# of only the run_detector_batch command-line tool -- see the "Load
# MegaDetector once" cell below for why.
from megadetector.detection import run_detector

# Create a hidden root window for file dialogs
root = tk.Tk()
root.withdraw()
root.attributes('-topmost', True)

print("All imports successful.")


## Configuration

Set up paths and detection parameters.

In [ ]:
# Model configuration
#
# Pointing directly at a shared, machine-wide model file instead of the "MDV5A"
# shorthand. The shorthand triggers an auto-download to each Windows profile's
# per-user %TEMP% cache the first time that profile runs the notebook -- this is
# what caused a CERTIFICATE_VERIFY_FAILED error under Shawn's account (the model
# was never cached there). Pointing to an explicit path skips the per-user cache
# and the download entirely, for every user, every time.
# See Tarazed-Session-Log.docx for the full diagnostic trail. The session logs
# are kept on Tarazed itself (not in this repo).
MODEL_PATH = r"C:\ProgramData\megadetector_models\md_v5a.0.1.pt"

if not os.path.exists(MODEL_PATH):
    raise FileNotFoundError(
        f"Shared model file not found at {MODEL_PATH}. "
        "This notebook expects the model to already be staged at this shared "
        "location -- see Tarazed-Session-Log.docx (on Tarazed) for setup steps."
    )

MODEL_NAME = MODEL_PATH  # kept as MODEL_NAME so the rest of the notebook is unchanged

# Detection parameters
ANIMAL_CATEGORY_ID = 1
# Animal confidence threshold (0.0 to 1.0), used for the animal checks and box drawing.
# 0.2 is the typical threshold Dan Morris recommends for MDv5a (it's the
# `typical_detection_threshold` MegaDetector itself lists for this model).
CONF_THRESH = 0.2

# ---------------------------------------------------------------------------
# FRAME SAMPLING -- how many frames per second of video MegaDetector looks at
# ---------------------------------------------------------------------------
# Our trail-camera videos are about 30 frames per second (~315 frames per video).
# Running MegaDetector on every frame is slow (~5.5 hours for 1,000 videos).
# Neighboring frames are nearly identical, so checking a few per second is
# usually enough to tell whether an animal is in the video.
#
#   SAMPLE_FPS = 2     check 2 frames per second (every 15th frame at 30 fps)
#                      -> about 15x less MegaDetector work than every frame
#   SAMPLE_FPS = 1     faster still, but more likely to miss a quick animal
#   SAMPLE_FPS = 5     slower, catches more brief appearances
#   SAMPLE_FPS = None  check EVERY frame (the old behavior, slowest)
#
# What you give up: an animal that is on screen for less than about
# 1/SAMPLE_FPS seconds (half a second at 2) can fall between checked frames and
# be missed. If a run misses something you know is there, raise this number
# or set it to None and re-run that folder.
#
# Other effects:
#   - Highlight videos (*_detected.mp4) contain only the checked frames, played
#     at SAMPLE_FPS, so they look choppier than the original but keep real timing.
#   - Frame numbers in detections.json are the ORIGINAL frame numbers, so
#     frame_00045.jpg is 45 / 30 = 1.5 seconds into the video.
SAMPLE_FPS = 2

# Verify MegaDetector is importable from the active environment
import megadetector  # noqa: E402
print(f"MegaDetector found: {megadetector.__file__}")
print(f"Model: {MODEL_NAME}")
print(f"Confidence threshold: {CONF_THRESH}")
print(f"Frames sampled per second: {SAMPLE_FPS if SAMPLE_FPS else 'all frames'}")

## Select Directories

Choose your input folder (containing videos) and output folder (where results will be saved).

In [ ]:
# Select input folder containing video files
input_folder = fd.askdirectory(title="Select folder containing video files")
if not input_folder:
    raise ValueError("No input folder selected")

DATA_DIR = Path(input_folder)
print("Input folder:", DATA_DIR)

In [ ]:
# Select output folder for all results
output_folder = fd.askdirectory(title="Select folder for output (detections, videos, etc.)")
if not output_folder:
    raise ValueError("No output folder selected")

OUTPUT_DIR = Path(output_folder)
OUTPUT_DIR.mkdir(exist_ok=True)
print("Output folder:", OUTPUT_DIR)

## Helper Functions

In [ ]:
def find_all_videos(root):
    """Find all video files recursively."""
    exts = (".mp4", ".mov", ".avi", ".MP4", ".MOV", ".AVI")
    videos = []
    for r, _, files in os.walk(root):
        for f in files:
            # Skip this notebook's own output videos (*_detected.mp4), in case the
            # output folder is inside the input folder and the notebook is re-run.
            if f.endswith(exts) and not Path(f).stem.endswith("_detected"):
                videos.append(Path(r) / f)
    return sorted(videos)

all_videos = find_all_videos(DATA_DIR)
print(f"Found {len(all_videos)} videos")

# Show first 20 videos
for v in all_videos[:20]:
    print(f"  {v.name}")
    
if len(all_videos) > 20:
    print(f"  ... and {len(all_videos) - 20} more")

In [ ]:
def resolve_frame_path(frames_dir, file_field):
    """Resolve the path to a frame file from MegaDetector JSON."""
    p = Path(file_field)
    if p.is_absolute() or p.exists():
        return p
    return frames_dir / p.name

## Process Videos

This cell processes each video individually:
1. Extract frames (every frame, or `SAMPLE_FPS` frames per second of video -- see Configuration)
2. Run MegaDetector
3. Draw bounding boxes on frames with detections
4. Create output video (only for videos WITH animal detections)
5. Clean up temporary frames

**Output style (differs from the Linux video notebook, on purpose):** each output video is an
**animal-only highlight reel** -- only frames with an animal at or above `CONF_THRESH` are written,
with boxes drawn. Videos with no animal get no output video. (The Linux notebook
`Batch-Video_Processing_2026.ipynb` instead writes the **full** video, with boxes on the animal frames.)

Videos that fail (no frames extracted, MegaDetector error, frame/result mismatch, unreadable
first frame) are counted and listed separately in the summary, with the reason, so a broken video
isn't mistaken for one with no animals.

**Frame sampling (`SAMPLE_FPS`, set in Configuration):** MegaDetector only looks at
`SAMPLE_FPS` frames per second of each video (default 2, i.e. every 15th frame of a 30 fps video)
instead of every frame. This makes a run roughly 15x less work. The trade-off is that an animal
on screen for less than about half a second can be missed. Set `SAMPLE_FPS = None` to check every
frame.

**Cameras that split videos into subfolders** (e.g. `DCIM\100_BTCF`, `DCIM\101_BTCF`, ... that
all reuse the names `IMG_0001.MP4`, `IMG_0002.MP4`, ...): select the whole camera folder (e.g.
`DCIM`) as the input. The output folder mirrors the same subfolders, so
`100_BTCF\IMG_0001` and `101_BTCF\IMG_0001` get separate results and never overwrite each other.
The summary at the end lists videos with their subfolder for the same reason.

**Note:** You can test with a few videos first by uncommenting the test line in the loop.

In [ ]:
import sys
print(sys.executable)

## Load MegaDetector once (speedup, 2026-09-13)

Previously, the batch loop below called the `run_detector_batch` **command-line tool** via
`subprocess.run(...)` once per video -- each call started a fresh Python process and reloaded the
model from disk from scratch (measured on Altair: ~2.5s fixed load time, plus a GPU warm-up ramp,
paid again on every single video -- roughly 20-25% of total wall-clock time on a 9-video test
batch). Loading the model once here, and calling its Python API
(`generate_detections_one_image()`) directly inside the loop instead of shelling out, removes that
repeated cost. Verified against real data on Altair (via a headless test copy of this notebook's
logic) before being applied here -- see the metrics log for details.

In [ ]:
DETECTOR = run_detector.load_detector(MODEL_NAME)
print("Model loaded once, ready to reuse across all videos.")

In [ ]:
# helper: check if video has any animal detections (any frame, whole video)
def video_has_any_animal(images, conf_thresh=0.0):
    for img in images:
        for det in img.get("detections", []):
            if int(det["category"]) == ANIMAL_CATEGORY_ID and det["conf"] >= conf_thresh:
                return True
    return False

# helper: check if a single frame has an animal detection (used to filter frames when restitching video)
def frame_has_animal(img, conf_thresh):
    for det in img.get("detections", []):
        if int(det["category"]) == ANIMAL_CATEGORY_ID and det["conf"] >= conf_thresh:
            return True
    return False

batch_start = time.perf_counter()

videos_processed = 0
videos_with_animals = 0
videos_skipped = 0
videos_with_animals_list = []
videos_skipped_list = []
videos_failed = 0
videos_failed_list = []   # "name: reason" -- failures, kept separate from "no animals"

for video_path in all_videos:
#for video_path in all_videos[:3]:  # TESTING: Uncomment to process only first 3 videos

    print("\n" + "="*50)
    print("Processing:", video_path.relative_to(DATA_DIR))
    print("="*50)

    # ----------------------------------------
    # Per-video output paths
    # ----------------------------------------
    # Derived from the video's path *relative to DATA_DIR*, not just its
    # filename stem -- two videos with the same filename in different
    # subfolders (e.g. different camera/location folders) would otherwise
    # map to the same output dir and silently overwrite each other's
    # detection results.
    video_stem = video_path.stem
    rel = video_path.relative_to(DATA_DIR)
    video_out_dir = OUTPUT_DIR / rel.with_suffix("")

    frames_dir = video_out_dir / "frames"
    json_path = video_out_dir / "detections.json"
    out_video = video_out_dir / f"{video_stem}_detected.mp4"

    video_out_dir.mkdir(parents=True, exist_ok=True)

    print("Frames:", frames_dir)
    print("JSON:", json_path)
    print("Output:", out_video)

    # ----------------------------------------
    # Clean frames dir
    # ----------------------------------------
    if frames_dir.exists():
        shutil.rmtree(frames_dir)
    frames_dir.mkdir()

    # ----------------------------------------
    # Extract frames
    # ----------------------------------------
    print("\n[1/4] Extracting frames...")
    cap = cv2.VideoCapture(str(video_path))
    fps = cap.get(cv2.CAP_PROP_FPS)

    # Keep every `step`-th frame (SAMPLE_FPS per second of video). Frame files keep
    # their original frame number, so frame_00045.jpg is still 45/fps seconds in.
    # cap.grab() advances past a frame without converting or saving it.
    if SAMPLE_FPS and fps > 0:
        step = max(1, round(fps / SAMPLE_FPS))
    else:
        step = 1

    frame_idx = 0     # frames read from the video
    frames_saved = 0  # frames saved for MegaDetector
    while True:
        if frame_idx % step == 0:
            ret, frame = cap.read()
            if not ret:
                break
            cv2.imwrite(
                str(frames_dir / f"frame_{frame_idx:05d}.jpg"),
                frame
            )
            frames_saved += 1
        elif not cap.grab():
            break
        frame_idx += 1

    cap.release()
    if step == 1:
        print(f"  Extracted {frames_saved} frames @ {fps:.2f} FPS")
    else:
        print(f"  Read {frame_idx} frames @ {fps:.2f} FPS; kept 1 of every {step} "
              f"({frames_saved} frames) for MegaDetector")

    if frames_saved == 0:
        print("  No frames extracted, skipping video")
        videos_failed += 1
        videos_processed += 1
        videos_failed_list.append(f"{rel}: no frames extracted")
        continue

    # ----------------------------------------
    # Run MegaDetector (in-process, using the model loaded once in the
    # "Load MegaDetector once" cell above -- no per-video reload/warm-up cost)
    # ----------------------------------------
    print("\n[2/4] Running MegaDetector...")
    frame_files = sorted(frames_dir.glob("frame_*.jpg"))
    images = []
    detection_failed = None
    for frame_path in frame_files:
        img_bgr = cv2.imread(str(frame_path))
        if img_bgr is None:
            images.append({"file": frame_path.name, "failure": "Failure image access"})
            continue
        try:
            # MegaDetector expects RGB; cv2.imread returns BGR.
            img_rgb = cv2.cvtColor(img_bgr, cv2.COLOR_BGR2RGB)
            # Keep detections down to MegaDetector's standard output threshold (0.005),
            # the same cutoff run_detector_batch applies. Without it, the Python API keeps
            # everything down to 0.00001 -- ~98% noise and much larger JSON files.
            images.append(DETECTOR.generate_detections_one_image(
                img_rgb,
                image_id=frame_path.name,
                detection_threshold=run_detector.DEFAULT_OUTPUT_CONFIDENCE_THRESHOLD,
            ))
        except Exception as e:
            detection_failed = str(e)
            break

    if detection_failed is not None:
        print("  MegaDetector failed:")
        print(f"  {detection_failed}")
        shutil.rmtree(frames_dir)
        videos_failed += 1
        videos_processed += 1
        videos_failed_list.append(f"{rel}: MegaDetector error: {detection_failed}")
        continue

    # Save the same {"images": [...]} shape run_detector_batch would have
    # written, so nothing downstream (or anyone inspecting the JSON by hand)
    # sees a difference.
    with open(json_path, "w") as f:
        json.dump({"images": images}, f)
    print("  Detection complete")

    # ----------------------------------------
    # Load detections
    # ----------------------------------------
    print(f"  Loaded {len(images)} detection results")

    if len(images) != frames_saved:
        print(f"  Frame/JSON mismatch ({frames_saved} frames vs {len(images)} results), skipping")
        shutil.rmtree(frames_dir)
        videos_failed += 1
        videos_processed += 1
        videos_failed_list.append(f"{rel}: frame/result mismatch ({frames_saved} vs {len(images)})")
        continue

    # --------------------------------------------------
    # NEW: Skip videos with NO animal detections
    # --------------------------------------------------
    print("\n[3/4] Checking for animal detections...")
    if not video_has_any_animal(images, CONF_THRESH):
        print(f"  No animal detections (conf >= {CONF_THRESH}) -- skipping output video")
        shutil.rmtree(frames_dir)
        videos_skipped += 1
        videos_processed += 1
        videos_skipped_list.append(str(rel))
        continue

    print("  Animals detected.")

    # ----------------------------------------
    # Initialize output video
    # ----------------------------------------
    print("\n[4/4] Creating output video (animal frames only, with bounding boxes)...")
    first_frame_path = resolve_frame_path(frames_dir, images[0]["file"])
    first_frame = cv2.imread(str(first_frame_path))

    if first_frame is None:
        print("  Cannot read first frame, skipping")
        shutil.rmtree(frames_dir)
        videos_failed += 1
        videos_processed += 1
        videos_failed_list.append(f"{rel}: cannot read first frame")
        continue

    h, w, _ = first_frame.shape

    out = cv2.VideoWriter(
        str(out_video),
        cv2.VideoWriter_fourcc(*"mp4v"),
        fps / step,  # kept frames at their real spacing (= fps when every frame is used)
        (w, h),
    )

    # --------------------------------------------------
    # NEW: Skip frames with NO animal detections
    # (only frames that pass this check get written below --
    # this is what makes the output video an animal-only
    # highlight reel instead of the full original video)
    # --------------------------------------------------
    written = 0
    frames_skipped_no_animal = 0
    detections_drawn = 0

    for img in images:
        if not frame_has_animal(img, CONF_THRESH):
            frames_skipped_no_animal += 1
            continue

        frame_path = resolve_frame_path(frames_dir, img["file"])
        frame = cv2.imread(str(frame_path))
        if frame is None:
            frames_skipped_no_animal += 1
            continue

        # draw bounding boxes (animal only)
        for det in img.get("detections", []):
            if int(det["category"]) != ANIMAL_CATEGORY_ID:
                continue
            if det["conf"] < CONF_THRESH:
                continue

            x, y, bw, bh = det["bbox"]
            x1 = int(x * w)
            y1 = int(y * h)
            x2 = int((x + bw) * w)
            y2 = int((y + bh) * h)

            cv2.rectangle(frame, (x1, y1), (x2, y2), (0, 255, 0), 2)
            cv2.putText(
                frame,
                f"{det['conf']:.2f}",
                (x1, max(y1 - 5, 10)),
                cv2.FONT_HERSHEY_SIMPLEX,
                0.5,
                (0, 255, 0),
                1,
            )
            detections_drawn += 1

        out.write(frame)
        written += 1

    out.release()
    print(f"  Wrote {written} animal frames with {detections_drawn} bounding boxes")
    print(f"  Skipped {frames_skipped_no_animal} frames with no animal detection")
    print(f"  Output video: {out_video}")

    videos_with_animals += 1
    videos_processed += 1
    videos_with_animals_list.append(str(rel))

    # ----------------------------------------
    # Cleanup frames
    # ----------------------------------------
    shutil.rmtree(frames_dir)
    print("  Cleaned up temporary frames")

print("\n" + "="*50)
print("PROCESSING COMPLETE")
print("="*50)


## Processing Summary

In [ ]:
batch_end = time.perf_counter()
elapsed = batch_end - batch_start

print("\n" + "="*50)
print("BATCH PROCESSING SUMMARY")
print("="*50)
print(f"Total videos found     : {len(all_videos)}")
print(f"Videos processed       : {videos_processed}")
print(f"Videos with animals    : {videos_with_animals}")
print(f"Videos skipped (no animals): {videos_skipped}")
print(f"Videos FAILED          : {videos_failed}")
print(f"Frames sampled per sec : {SAMPLE_FPS if SAMPLE_FPS else 'all frames'}")
print(f"Total elapsed time     : {elapsed:.2f} seconds ({elapsed/60:.1f} minutes)")
print(f"Average per video      : {elapsed / max(videos_processed, 1):.2f} seconds")

print("\n" + "-"*50)
print("Videos WITH animal detections:")
print("-"*50)
if videos_with_animals_list:
    for v in videos_with_animals_list:
        print(f"  - {v}")
else:
    print("  (none)")

print("\n" + "-"*50)
print("Videos with NO animal detections:")
print("-"*50)
if videos_skipped_list:
    for v in videos_skipped_list:
        print(f"  - {v}")
else:
    print("  (none)")

print("\n" + "-"*50)
print("Videos that FAILED (not processed -- check these):")
print("-"*50)
if videos_failed_list:
    for v in videos_failed_list:
        print(f"  - {v}")
else:
    print("  (none)")

print("\n" + "="*50)
print(f"Output directory: {OUTPUT_DIR}")
print("="*50)
